In [67]:
import pandas as pd
df=pd.read_csv('dataforbotescalationsystem.csv')
print(df.head())

  conversation_id                                           messages  label  \
0        conv_001  User: Hi, did my refund go through? Order #773...      0   
1        conv_002  User: My package was delivered to the wrong ad...      1   
2        conv_003  User: I've been waiting 2 weeks for my refund....      1   
3        conv_004  User: My login stopped working after your upda...      1   
4        conv_005  User: Hi, I ordered 3 items but only got 2. Sh...      0   

                           reasons  severity  
0             resolved,smooth_flow      0.52  
1             frustration,fallback      0.72  
2  frustration,repetition,fallback      0.89  
3  frustration,repetition,fallback      0.89  
4               resolved,clarified      0.54  


In [68]:
import nltk
nltk.download('vader_lexicon')

[nltk_data] Error loading vader_lexicon: <urlopen error [SSL:
[nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed:
[nltk_data]     unable to get local issuer certificate (_ssl.c:1032)>


False

In [69]:
df.dropna(inplace=True)
df['messages'] = df['messages'].str.strip()
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 230 entries, 0 to 229
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   conversation_id  230 non-null    object 
 1   messages         230 non-null    object 
 2   label            230 non-null    int64  
 3   reasons          230 non-null    object 
 4   severity         230 non-null    float64
dtypes: float64(1), int64(1), object(3)
memory usage: 9.1+ KB
None


In [70]:
def parse_conversation(text):
    turns = text.split("||")
    parsed = []
    
    for t in turns:
        if ":" in t:
            role, msg = t.split(":", 1)
            parsed.append((role.strip(), msg.strip()))
    
    return parsed

df['parsed'] = df['messages'].apply(parse_conversation)
print(df['parsed'].iloc[0])

[('User', 'Hi, did my refund go through? Order #7734.'), ('Bot', 'Let me check… Yes, a refund of Rs.89.99 was processed on April 2nd. It should appear on your statement by April 7th.'), ('User', "Okay cool. I'll keep an eye out. Thanks."), ('Bot', "You're welcome! Feel free to reach out if it doesn't show up.")]


In [71]:
def split_roles(parsed):
    user_msgs = [m for r, m in parsed if r == "User"]
    bot_msgs = [m for r, m in parsed if r == "Bot"]
    return user_msgs, bot_msgs

df['user_msgs'], df['bot_msgs'] = zip(*df['parsed'].apply(split_roles))


In [72]:
df['num_turns'] = df['parsed'].apply(len)
df['num_user_msgs'] = df['user_msgs'].apply(len)
df['num_bot_msgs'] = df['bot_msgs'].apply(len)
df['user_bot_ratio'] = df['num_user_msgs'] / (df['num_bot_msgs'] + 1e-5)
print(df[['num_turns', 'num_user_msgs', 'num_bot_msgs', 'user_bot_ratio']].head())

   num_turns  num_user_msgs  num_bot_msgs  user_bot_ratio
0          4              2             2        0.999995
1          9              5             4        1.249997
2          7              4             3        1.333329
3          8              4             4        0.999998
4          7              4             3        1.333329


In [73]:
print(df.groupby('label')['num_turns'].mean())

label
0    5.217391
1    8.184783
Name: num_turns, dtype: float64


In [74]:
import nltk
nltk.download('vader_lexicon')

[nltk_data] Error loading vader_lexicon: <urlopen error [SSL:
[nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed:
[nltk_data]     unable to get local issuer certificate (_ssl.c:1032)>


False

In [75]:
from nltk.sentiment import SentimentIntensityAnalyzer
sia = SentimentIntensityAnalyzer()

def sentiment_scores(messages):
    return [sia.polarity_scores(m)['compound'] for m in messages]

df['user_sentiments'] = df['user_msgs'].apply(sentiment_scores)

In [76]:
print(df['user_msgs'].iloc[0])
print(df['user_sentiments'].iloc[8])

['Hi, did my refund go through? Order #7734.', "Okay cool. I'll keep an eye out. Thanks."]
[0.0, 0.1027, 0.0772, -0.1139]


In [77]:
df['avg_sentiment'] = df['user_sentiments'].apply(
    lambda x: sum(x)/len(x) if len(x) > 0 else 0
)
df['min_sentiment'] = df['user_sentiments'].apply(
    lambda x: min(x) if len(x) > 0 else 0
)
def sentiment_trend(scores):
    if len(scores) < 2:
        return 0
    return scores[-1] - scores[0]

df['sentiment_trend'] = df['user_sentiments'].apply(sentiment_trend)

In [78]:
print(df.groupby('label')['avg_sentiment'].mean())
print(df.groupby('label')['min_sentiment'].mean())

label
0    0.187416
1   -0.116296
Name: avg_sentiment, dtype: float64
label
0   -0.101357
1   -0.410716
Name: min_sentiment, dtype: float64


In [79]:
import numpy as np
import sklearn
print("Installed successfully")
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def repetition_score(user_msgs):
    if len(user_msgs) < 2:
        return 0
    
    vectorizer = TfidfVectorizer().fit(user_msgs)
    vectors = vectorizer.transform(user_msgs)
    
    sim_scores = []
    
    for i in range(1, len(user_msgs)):
        sim = cosine_similarity(vectors[i], vectors[i-1])[0][0]
        sim_scores.append(sim)
    
    return max(sim_scores)  # strongest repetition

df['repetition_score'] = df['user_msgs'].apply(repetition_score)

Installed successfully


In [80]:
fallback_phrases = [
    "i don't understand",
    "sorry",
    "can you rephrase",
    "i didn't get that",
    "unable to help",
    "not sure",
    "cannot help",
    "please clarify"
]
def fallback_count(bot_msgs):
    count = 0
    
    for msg in bot_msgs:
        msg_lower = msg.lower()
        if any(p in msg_lower for p in fallback_phrases):
            count += 1
    
    return count

df['fallback_count'] = df['bot_msgs'].apply(fallback_count)

In [81]:
print(df.groupby('label')['fallback_count'].mean())

label
0    0.326087
1    0.836957
Name: fallback_count, dtype: float64


In [82]:
frustration_keywords = [
    "frustrated", "angry", "not working", "still",
    "again", "why", "worst", "hate", "useless",
    "issue", "problem", "error","person","representative", "agent","real","human"
]

def frustration_score(user_msgs):
    score = 0
    
    for msg in user_msgs:
        msg_lower = msg.lower()
        if any(k in msg_lower for k in frustration_keywords):
            score += 1
    
    return score

df['frustration_score'] = df['user_msgs'].apply(frustration_score)

In [83]:
generic_responses = [
    "i am here to help",
    "please clarify",
    "can you provide more details",
    "let me check",
    "i will assist you",
    "thanks for reaching out"
]

def generic_ratio(bot_msgs):
    if len(bot_msgs) == 0:
        return 0
    
    generic_count = 0
    
    for msg in bot_msgs:
        msg_lower = msg.lower()
        if any(g in msg_lower for g in generic_responses):
            generic_count += 1
    
    return generic_count / len(bot_msgs)

df['generic_response_ratio'] = df['bot_msgs'].apply(generic_ratio)


In [84]:
features = [
    'num_turns',
    'num_user_msgs',
    'num_bot_msgs',
    'user_bot_ratio',
    'avg_sentiment',
    'min_sentiment',
    'sentiment_trend',
    'fallback_count',
    'frustration_score',
    'generic_response_ratio'
]

X = df[features]
y = df['label']

print (X.head())
print(y.head())


   num_turns  num_user_msgs  num_bot_msgs  user_bot_ratio  avg_sentiment  \
0          4              2             2        0.999995       0.363450   
1          9              5             4        1.249997      -0.341840   
2          7              4             3        1.333329      -0.361425   
3          8              4             4        0.999998       0.028425   
4          7              4             3        1.333329       0.252250   

   min_sentiment  sentiment_trend  fallback_count  frustration_score  \
0         0.0000           0.7269               0                  0   
1        -0.6093           0.0998               1                  0   
2        -0.7233          -0.3621               1                  2   
3        -0.2263           0.5663               1                  1   
4         0.0000           0.3612               1                  0   

   generic_response_ratio  
0                0.500000  
1                0.000000  
2                0.000000 

In [85]:
# Boost signals directly in existing features
X['sentiment_trend'] = X['sentiment_trend'].apply(lambda x: -x if x < 0 else 0)  # treat drops as positive signal

/var/folders/x2/q62271lx743cz_w1dz6rr0h80000gn/T/ipykernel_1860/2257001922.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['sentiment_trend'] = X['sentiment_trend'].apply(lambda x: -x if x < 0 else 0)  # treat drops as positive signal


In [86]:
# Split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train model
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(class_weight='balanced', max_iter=1000)
model.fit(X_train_scaled, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [87]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


Accuracy: 0.9347826086956522
              precision    recall  f1-score   support

           0       1.00      0.89      0.94        28
           1       0.86      1.00      0.92        18

    accuracy                           0.93        46
   macro avg       0.93      0.95      0.93        46
weighted avg       0.94      0.93      0.94        46



In [88]:
print(hasattr(model, "coef_"))

True


In [89]:
print(df.groupby('label').mean(numeric_only=True))

       severity  num_turns  num_user_msgs  num_bot_msgs  user_bot_ratio  \
label                                                                     
0      0.552971   5.217391       2.717391      2.500000        1.084536   
1      0.816522   8.184783       4.510870      3.673913        1.236410   

       avg_sentiment  min_sentiment  sentiment_trend  repetition_score  \
label                                                                    
0           0.187416      -0.101357         0.528997          0.054889   
1          -0.116296      -0.410716         0.094720          0.144745   

       fallback_count  frustration_score  generic_response_ratio  
label                                                             
0            0.326087           0.202899                0.054348  
1            0.836957           0.989130                0.038043  


In [100]:
def predict_escalation(conversation_text, model, scaler, threshold=0.7):
    
    # --- Parse ---
    parsed = parse_conversation(conversation_text)
    user_msgs, bot_msgs = split_roles(parsed)
    
    # --- Feature computation ---
    num_turns = len(parsed)
    num_user_msgs = len(user_msgs)
    num_bot_msgs = len(bot_msgs)
    user_bot_ratio = num_user_msgs / (num_bot_msgs + 1e-5)
    
    # Sentiment
    sentiments = sentiment_scores(user_msgs)
    avg_sentiment = sum(sentiments)/len(sentiments) if sentiments else 0
    min_sentiment = min(sentiments) if sentiments else 0
    sentiment_trend = sentiments[-1] - sentiments[0] if len(sentiments) > 1 else 0
    
    # Repetition
    rep_score = repetition_score(user_msgs)
    
    # Fallback
    fb_count = fallback_count(bot_msgs)
    
    # Frustration
    fr_score = frustration_score(user_msgs)
    
    # Generic responses
    gen_ratio = generic_ratio(bot_msgs)
    
    # --- Feature vector ---
    features = [[
        num_turns,
        num_user_msgs,
        num_bot_msgs,
        user_bot_ratio,
        avg_sentiment,
        min_sentiment,
        sentiment_trend,
        fb_count,
        fr_score,
        gen_ratio
    ]]
    
    # --- Scale ---
    features_scaled = scaler.transform(features)
    
    # --- Predict probability ---
    prob = model.predict_proba(features_scaled)[0][1]
    
    # --- Decision ---
    decision = 1 if prob >= threshold else 0
    
    return prob, decision

In [91]:
print(hasattr(model, "coef_"))

True


In [92]:
conv_fromrealsample = (
    "Bot: Hi! I am your automated assistant. Please enter your 10-digit Order ID so I can look up your details. || "
    "User: 1234567890 || "
    "Bot: I'm listening. Could you give me a little more context? Are you dealing with a delayed delivery, a missing item, or something else entirely? || "
    "User: an item was missing in my order || "
    "Bot: Oh no, I am so sorry to hear that. What item was missing? || "
    "User: dal|| "
    "Bot: Thank you. I see your order. To start, could you tell me exactly which items were missing or damaged? || "
    "User: dal was missing. this is unacceptable i want to talk to cutomer care || "
    "Bot: Got it. I have noted that your refund will be issued soon || "
    "User: no connect me to a person || "
    "Bot: your refund is on the way || "
    "User: No i want to be connected to a human|| "
    "Bot: Thank you for the information. I have analyzed your request and checked with the restaurant's dispatch system. Unfortunately, the restaurant has marked all items as verified and packed. Because of this, I cannot process a refund or replacement for this order. Is there anything else I can assist with? || "
    "User: no you are of no use || "
)

In [93]:
print(predict_escalation(conv_fromrealsample, model, scaler, threshold=0.5))

(np.float64(0.9981035920833019), 1)


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [94]:
conv1 = """User: Hi || Bot: Hello || User: I need help with login || Bot: Sure, reset your password || User: Thanks || Bot: You're welcome"""

print(predict_escalation(conv1, model, scaler))

(np.float64(0.09058188325593855), 0)


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [95]:
conv=""" "User: Hey there! How’s it going? || "
    "Bot: Hello! I’m good, how about you? || "
    "User: I’m fine, thanks for asking this is the 3rd time i am facing this issueee|| "
    "Bot: Glad to hear that!"""

In [96]:
print(predict_escalation(conv, model, scaler))

(np.float64(1.3496823651369348e-07), 0)


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [97]:
import pickle

pickle.dump(model, open("escalation_model.pkl","wb"))
pickle.dump(scaler, open("scaler.pkl","wb"))

